In [1]:
!pip install ultralytics -q

from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 5.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
DATA_ROOT = "/content/drive/MyDrive/BARS_yolo"          # dossier qui contient images/ et labels/
ORIG_YAML = "/content/drive/MyDrive/bars_yolo_seg.yaml" # ton yaml original


In [5]:
import numpy as np

def parse_polygon(coords_flat):
    """Liste [x0,y0,x1,y1,...] → array (N,2)"""
    pts = np.array(coords_flat, dtype=float).reshape(-1, 2)
    return pts

def top_mid_bottom_mid(pts):
    """
    Retourne (top_mid, bot_mid) où :
      - top_mid = milieu des points ayant y minimal (côté haut)
      - bot_mid = milieu des points ayant y maximal (côté bas)
    On sélectionne les 2 points les plus proches du min/max de y.
    """
    ys = pts[:, 1]

    # Top : 2 points avec le plus petit y
    top_idx = np.argsort(ys)[:2]
    top_mid = pts[top_idx].mean(axis=0)

    # Bottom : 2 points avec le plus grand y
    bot_idx = np.argsort(ys)[-2:]
    bot_mid = pts[bot_idx].mean(axis=0)

    return top_mid, bot_mid

def centerline_triangle(pts, half_width=0.005):
    """
    Construit le triangle centerline :
      - sommet A : milieu du côté haut
      - sommet B : milieu côté bas décalé à gauche de half_width
      - sommet C : milieu côté bas décalé à droite de half_width
    """
    top_mid, bot_mid = top_mid_bottom_mid(pts)

    A = top_mid
    B = np.array([bot_mid[0] - half_width, bot_mid[1]])
    C = np.array([bot_mid[0] + half_width, bot_mid[1]])

    return A, B, C

def format_triangle_line(A, B, C, class_id=0):
    coords = [A[0], A[1], B[0], B[1], C[0], C[1]]
    coords_str = " ".join(f"{v:.6f}" for v in coords)
    return f"{class_id} {coords_str}"

print("Utilitaires chargés ✓")

Utilitaires chargés ✓


In [6]:
def is_centerline(coords):
    """Un triangle = exactement 3 points = 6 coordonnées."""
    return len(coords) == 6

In [ ]:
def process_txt(filepath, dry_run=True):
    with open(filepath, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    new_lines = []
    centerline_line = None


    for line in lines:
      parts = line.split()
      cls = int(parts[0])
      coords = list(map(float, parts[1:]))
      if cls == 0 and is_centerline(coords):
          print(f"⏭️  Déjà traité, ignoré : {filepath}")
          return None

    for line in lines:
        parts = line.split()
        cls = int(parts[0])
        coords = list(map(float, parts[1:]))

        if cls == 0:
            # C'est la runway → calculer la centerline
            pts = parse_polygon(coords)
            A, B, C = centerline_triangle(pts, half_width=0.005)
            centerline_line = format_triangle_line(A, B, C, class_id=0)
            # Runway elle-même → reroulée en classe 1
            new_lines.append(f"1 " + " ".join(f"{v:.6f}" for v in coords))
        else:
            # Autres classes décalées de +1
            new_lines.append(f"{cls + 1} " + " ".join(f"{v:.6f}" for v in coords))

    # Centerline en tête
    if centerline_line:
        output = [centerline_line] + new_lines
    else:
        output = new_lines  # pas de runway trouvée, on laisse tel quel

    if dry_run:
        print(f"\n=== {filepath} ===")
        for l in output:
            print(l)
    else:
        with open(filepath, "w") as f:
            f.write("\n".join(output) + "\n")

    return output

# Test sur UN fichier — remplace par un vrai chemin
TEST_FILE = "/content/drive/MyDrive/BARS_yolo/labels/train/CYOW_55.txt"
process_txt(TEST_FILE, dry_run=True)


=== /content/drive/MyDrive/BARS_yolo/labels/train/CYOW_55.txt ===
0 0.441470 0.334433 0.441833 0.409241 0.451833 0.409241
1 0.442450 0.409058 0.451217 0.409424 0.444513 0.334067 0.438428 0.334800
3 0.442249 0.394625 0.442326 0.395997 0.443702 0.396016 0.443616 0.394639
3 0.447907 0.394684 0.448019 0.396074 0.449428 0.396093 0.449308 0.394699
2 0.442655 0.406637 0.442771 0.408720 0.450828 0.409051 0.450642 0.406929


['0 0.441470 0.334433 0.441833 0.409241 0.451833 0.409241',
 '1 0.442450 0.409058 0.451217 0.409424 0.444513 0.334067 0.438428 0.334800',
 '3 0.442249 0.394625 0.442326 0.395997 0.443702 0.396016 0.443616 0.394639',
 '3 0.447907 0.394684 0.448019 0.396074 0.449428 0.396093 0.449308 0.394699',
 '2 0.442655 0.406637 0.442771 0.408720 0.450828 0.409051 0.450642 0.406929']

In [ ]:
import os
from pathlib import Path

LABELS_ROOT = "/content/BARS_yolo/label"  # contient train/, val/, test/ etc.

txt_files = list(Path(LABELS_ROOT).rglob("*.txt"))
print(f"{len(txt_files)} fichiers .txt trouvés")

no_runway = []


for fp in txt_files:
    result = process_txt(str(fp), dry_run=False)
    # Vérifier si aucune runway n'a été trouvée
    if result is None:
        continue
    if not any(l.startswith("0 ") for l in result):
        no_runway.append(str(fp))

print(f"\n✅ Traitement terminé.")
if no_runway:
    print(f"⚠️  {len(no_runway)} fichiers sans runway (classe 0) détectée :")
    for f in no_runway:
        print(" ", f)

10254 fichiers .txt trouvés
⏭️  Déjà traité, ignoré : /content/BARS_yolo/label/train/XPlane4_413.txt

✅ Traitement terminé.


In [ ]:
import yaml

with open("/content/drive/MyDrive/bars_yolo_seg.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/content/BARS_yolo"  # ← chemin local Colab

old_names = cfg.get("names", {})

if isinstance(old_names, list):
    old_names = {i: v for i, v in enumerate(old_names)}

new_names = {0: "centerline"}
for k, v in old_names.items():
    new_names[k + 1] = v

cfg["names"] = new_names
cfg["nc"] = len(new_names)

NEW_YAML = "/content/bars_yolo_seg_colab_binary.yaml"
with open(NEW_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

print("Nouveau YAML :")
print(open(NEW_YAML).read())




Nouveau YAML :
names:
  0: centerline
  1: runway
  2: threshold
  3: aiming
nc: 4
path: /content/BARS_yolo
test: images/test
train: images/train
val: images/val



In [ ]:
!cp -r "/content/drive/MyDrive/BARS_yolo/labels" "/content/BARS_yolo/labels"

cp: cannot create directory '/content/BARS_yolo/labels': No such file or directory


In [ ]:
!mkdir -p "/content/BARS_yolo/labels"


In [ ]:
!rsync -a --progress "/content/drive/MyDrive/BARS_yolo/labels/" "/content/BARS_yolo/labels/"

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
train/XPlane_VMMC_347.txt
             73 100%    0.09kB/s    0:00:00 (xfr#1159, to-chk=2499/10260)
train/XPlane_VMMC_35.txt
            352 100%    0.42kB/s    0:00:00 (xfr#1160, to-chk=2498/10260)
train/XPlane_VMMC_352.txt
             73 100%    0.00kB/s    0:00:00 (xfr#1161, to-chk=2497/10260)
train/XPlane_VMMC_353.txt
             73 100%    0.10kB/s    0:00:00 (xfr#1162, to-chk=2496/10260)
train/XPlane_VMMC_354.txt
            130 100%    0.17kB/s    0:00:00 (xfr#1163, to-chk=2495/10260)
train/XPlane_VMMC_355.txt
             73 100%    0.05kB/s    0:00:01 (xfr#1164, to-chk=2494/10260)
train/XPlane_VMMC_356.txt
             73 100%    0.00kB/s    0:00:00 (xfr#1165, to-chk=2493/10260)
train/XPlane_VMMC_357.txt
             73 100%    0.09kB/s    0:00:00 (xfr#1166, to-chk=2492/10260)
train/XPlane_VMMC_358.txt
             73 100%    0.00kB/s    0:00:00 (xfr#1167, to-chk=2491/10260)
train/XPlane_VMMC_36.tx

In [ ]:
!rsync -a --progress "/content/drive/MyDrive/BARS_yolo/images/" "/content/BARS_yolo/images/"

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
train/XPlane_VMMC_276.jpg
         92,366 100%   68.08kB/s    0:00:01 (xfr#7696, to-chk=2560/10260)
train/XPlane_VMMC_277.jpg
         93,887 100%  131.54kB/s    0:00:00 (xfr#7697, to-chk=2559/10260)
train/XPlane_VMMC_278.jpg
         95,341 100%   67.66kB/s    0:00:01 (xfr#7698, to-chk=2558/10260)
train/XPlane_VMMC_279.jpg
         96,769 100%  121.16kB/s    0:00:00 (xfr#7699, to-chk=2557/10260)
train/XPlane_VMMC_280.jpg
         98,758 100%   62.93MB/s    0:00:00 (xfr#7700, to-chk=2556/10260)
train/XPlane_VMMC_281.jpg
        100,802 100%  140.03kB/s    0:00:00 (xfr#7701, to-chk=2555/10260)
train/XPlane_VMMC_283.jpg
        100,823 100%   70.23kB/s    0:00:01 (xfr#7702, to-chk=2554/10260)
train/XPlane_VMMC_284.jpg
         99,080 100%  139.82kB/s    0:00:00 (xfr#7703, to-chk=2553/10260)
train/XPlane_VMMC_285.jpg
         96,364 100%   70.92kB/s    0:00:01 (xfr#7704, to-chk=2552/10260)
train/XPlane_VMMC_286.

In [ ]:
#!ls "/content/BARS_yolo/labels" | wc -l  # nombre de sous-dossiers copiés

1


In [ ]:
!cp "/content/drive/MyDrive/BARS_yolo.zip" "/content/"

In [ ]:
!rm -rf "/content/BARS_yolo/labels"


In [ ]:
!unzip -q "/content/label.zip" -d "/content/BARS_yolo/"

In [ ]:
#!ls /content/BARS_yolo/BARS_yolo/

ls: cannot access '/content/BARS_yolo/BARS_yolo/': No such file or directory


In [ ]:
#!unzip -l "/content/BARS_yolo.zip" | head -30

Archive:  /content/BARS_yolo.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
      141  2026-03-25 23:47   BARS_yolo
---------                     -------
      141                     1 file


In [ ]:
import os
from pathlib import Path

LABELS_ROOT = "/content/BARS_yolo/label"
LABELS2_ROOT = "/content/BARS_yolo/label2"

txt_files = list(Path(LABELS_ROOT).rglob("*.txt"))
print(f"{len(txt_files)} fichiers .txt trouvés")

for fp in txt_files:
    with open(fp, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    # Garder uniquement les lignes de classe 0
    kept = [l for l in lines if l.split()[0] == "0"]

    # Créer le même sous-dossier dans labels2
    relative = fp.relative_to(LABELS_ROOT)
    out_path = Path(LABELS2_ROOT) / relative
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w") as f:
        f.write("\n".join(kept) + "\n" if kept else "")

print(f"✅ Terminé — fichiers écrits dans {LABELS2_ROOT}")

10254 fichiers .txt trouvés
✅ Terminé — fichiers écrits dans /content/BARS_yolo/label2


In [ ]:
from pathlib import Path

LABELS_ROOT = "/content/BARS_yolo/labels"
LABELS3_ROOT = "/content/BARS_yolo/labels3"

txt_files = list(Path(LABELS_ROOT).rglob("*.txt"))
print(f"{len(txt_files)} fichiers .txt trouvés")

for fp in txt_files:
    with open(fp, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    # Garder uniquement classes 0 (centerline) et 1 (runway)
    kept = [l for l in lines if l.split()[0] in ("0", "1")]

    relative = fp.relative_to(LABELS_ROOT)
    out_path = Path(LABELS3_ROOT) / relative
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w") as f:
        f.write("\n".join(kept) + "\n" if kept else "")

print(f"✅ Terminé — fichiers écrits dans {LABELS3_ROOT}")

10254 fichiers .txt trouvés
✅ Terminé — fichiers écrits dans /content/BARS_yolo/labels3


In [ ]:
import yaml

with open("/content/bars_yolo_seg.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/content/BARS_yolo"
cfg["names"] = {0: "centerline"}
cfg["nc"] = 1
cfg["train"] = "/content/BARS_yolo/images/train"
cfg["val"] = "/content/BARS_yolo/images/val"
cfg["test"] = "/content/BARS_yolo/images/test"

NEW_YAML = "/content/bars_yolo_centerline.yaml"
with open(NEW_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

print("Nouveau YAML :")
print(open(NEW_YAML).read())

Nouveau YAML :
names:
  0: centerline
nc: 1
path: /content/BARS_yolo
test: /content/BARS_yolo/images/test
train: /content/BARS_yolo/images/train
val: /content/BARS_yolo/images/val



In [ ]:
import yaml

with open("/content/drive/MyDrive/bars_yolo_seg.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/content/BARS_yolo"
cfg["names"] = {0: "centerline", 1: "runway"}
cfg["nc"] = 2
cfg["train"] = "/content/BARS_yolo/images/train"
cfg["val"] = "/content/BARS_yolo/images/val"
cfg["test"] = "/content/BARS_yolo/images/test"

NEW_YAML = "/content/bars_yolo_centerline_runway.yaml"
with open(NEW_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

print("Nouveau YAML :")
print(open(NEW_YAML).read())

Nouveau YAML :
names:
  0: centerline
  1: runway
nc: 2
path: /content/BARS_yolo
test: /content/BARS_yolo/images/test
train: /content/BARS_yolo/images/train
val: /content/BARS_yolo/images/val



In [ ]:
model = YOLO("yolo26n-seg.pt")  # Ultralytics va télécharger le poids si besoin

model.train(
    data="/content/bars_yolo_centerline.yaml",
    epochs=8,       # ou 5 pour un test rapide
    imgsz=640,       # ou 1024 si le GPU le permet
    batch=16,        # ajuste si OOM
    device=0,        # GPU Colab
    workers=4,       # 2–4 en général OK
    save_period=1
)

In [ ]:
model = YOLO("yolo26n-seg.pt")  # Ultralytics va télécharger le poids si besoin

model.train(
    data="/content/bars_yolo_centerline_runway.yaml",
    epochs=8,       # ou 5 pour un test rapide
    imgsz=640,       # ou 1024 si le GPU le permet
    batch=16,        # ajuste si OOM
    device=0,        # GPU Colab
    workers=4,       # 2–4 en général OK
    save_period=1
)

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bars_yolo_centerline_runway.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=8, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ed5a87e5c40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

In [ ]:
!ls -lh

total 5.6M
drwxr-xr-x 4 root root 4.0K Mar 26 05:28 BARS_yolo
-rw-r--r-- 1 root root  187 Mar 26 05:20 bars_yolo_seg.yaml
drwx------ 6 root root 4.0K Mar 26 05:16 drive
-rw-r--r-- 1 root root 5.6M Mar 26 05:19 label.zip
drwxr-xr-x 1 root root 4.0K Mar 23 13:34 sample_data


In [ ]:
from google.colab import files
files.upload()

In [ ]:
import shutil
shutil.make_archive("/content/labels", "zip", "/content/BARS_yolo/labels")

'/content/labels.zip'